# ENTREGA PARCIAL 1 - ALVARO FERNANDEZ


In [5]:
from dotenv import load_dotenv
import os, requests, uuid, json 

load_dotenv()

KEY_TRANSLATOR = os.getenv("TRAD_KEY1")
TRAD_ENDPOINT = os.getenv("ENDPOINT_TRAD")
KEY_SPEECH = os.getenv("SPEECH_KEY1")
SPEECH_ENDPOINT = os.getenv("ENDPOINT_SPEECH")
REGION = os.getenv("REGION")


print("Translator:", TRAD_ENDPOINT, REGION, bool(KEY_TRANSLATOR))
print("Speech    :", SPEECH_ENDPOINT, REGION, bool(KEY_SPEECH))

Translator: https://api.cognitive.microsofttranslator.com/ westeurope True
Speech    : https://westeurope.api.cognitive.microsoft.com/ westeurope True


In [ ]:
# Comandos de instalacion
# pip install python-dotenv
# pip install azure-cognitiveservices-speech
# pip install azure-ai-translation-text
# pip install azure-ai-textanalytics

# **1. TRADUCIR - API Y SDK**

### 1.1 TRADUCTOR API

In [6]:
# PRUEBA API REST TRANSLATOR 
print("PRUEBA API TRANSLATOR.")

# Configuracion
path = '/translate'
url = TRAD_ENDPOINT + path

params = {
    'api-version': '3.0',
    'from': 'es',
    'to': ['en', 'fr', 'de']
}

headers = {
    'Ocp-Apim-Subscription-Key': KEY_TRANSLATOR,
    'Ocp-Apim-Subscription-Region': REGION,
    'Content-type': 'application/json'
}

body = [{
    'text': 'Hola que tal, esto es una prueba'
}]

# Peticion
request = requests.post(url, params=params, headers=headers, json=body)
response = request.json()

if request.status_code == 200:
    print("Responde correctamente.")
    print("Respuesta completa:")
    print(json.dumps(response, sort_keys=True, ensure_ascii=False, indent=2, separators=(',', ': ')))
    
    # Traduccion 1
    traduccion = response[0]['translations'][0]['text']
    print(f"\n📝 Original: 'Hola que tal, esto es una prueba de traduccion con API'")
    print(f"📝 Traducción: '{traduccion}'")
    # Ejemplo mostrar todas las traducciones disponibles
    print("\nTraducciones disponibles:")
    for i in response: 
        for trad in i['translations']:
            print(f" - {trad['to']}: {trad['text']}")
else:
    print(f" Error: {request.status_code}")
    print(request.text)


PRUEBA API TRANSLATOR.
Responde correctamente.
Respuesta completa:
[
  {
    "translations": [
      {
        "text": "Hi how are you, this is a test",
        "to": "en"
      },
      {
        "text": "Bonjour, comment allez-vous, c’est un test",
        "to": "fr"
      },
      {
        "text": "Hallo, wie geht's dir, das ist ein Test",
        "to": "de"
      }
    ]
  }
]

📝 Original: 'Hola que tal, esto es una prueba de traduccion con API'
📝 Traducción: 'Hi how are you, this is a test'

Traducciones disponibles:
 - en: Hi how are you, this is a test
 - fr: Bonjour, comment allez-vous, c’est un test
 - de: Hallo, wie geht's dir, das ist ein Test


### 1.2 TRADUCTOR SDK

In [7]:
# PRUEBA SDK TRANSLATOR 
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential

print(" PRUEBA SDK TRANSLATOR.")

try:
    client = TextAnalyticsClient(
        endpoint=TRAD_ENDPOINT,
        credential=AzureKeyCredential(KEY_TRANSLATOR)
    )
    
    # Detectar idioma + traducir
    documents = ["Hola que tal, esto es una prueba de traduccion con SDK"]
    
    # Detectar idioma
    result = client.detect_language(documents, country_hint='ES')[0]
    print(f"Idioma detectado: {result.primary_language.name} (score: {result.primary_language.confidence_score:.2f})")
    
    # Traducir
    poller = client.begin_translate(documents, "en")
    translation_result = poller.result()
    
    print("✅ SDK Translator OK")
    print(f"Traducción: {translation_result[0][0].text}")
    
except Exception as e:
    print(f"❌ Error: {e}")


ModuleNotFoundError: No module named 'azure.ai.textanalytics'

# 2. **SPEECH - SDK Y API**
- Tema audio mejor con SDK, por permisos

### 2.1 TTS SDK

In [8]:
# PRUEBA SDK SPEECH 
import azure.cognitiveservices.speech as speechsdk

print("Probando SDK Speech (Text-to-Speech)...")

# Configurar SpeechConfig
speech_config = speechsdk.SpeechConfig(
    subscription=KEY_SPEECH,
    region=REGION
)

# Opcional: voz en español
speech_config.speech_synthesis_voice_name = "es-ES-ElviraNeural"

audio_config = speechsdk.audio.AudioOutputConfig(filename="salida_hola.wav")

# Sintetizar texto → audio
synthesizer = speechsdk.SpeechSynthesizer(speech_config=speech_config, audio_config=audio_config)

try:
    result = synthesizer.speak_text_async("Hola, esta es una prueba TTS con SDK").get()
    
    if result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
        print(" Archivo creado")
    else:
        print(f"Sintesis falló: {result.reason}")
        
except Exception as e:
    print(f"Error: {e}")


🔄 Probando SDK Speech (Text-to-Speech)...
 Archivo creado


### 2.2 STT SDK

In [17]:
# PRUEBA SDK SPEECH (Speech-to-Text)
print(" Probando SDK Speech (Speech-to-Text)...")

# Configurar SpeechConfig
speech_config = speechsdk.SpeechConfig(
    subscription=KEY_SPEECH,
    region=REGION
)
# Detectar entrada de micro. Si no se detecta entrada, usar archivo de audio de prueba
audio_config = speechsdk.audio.AudioConfig(use_default_microphone=True)
recognizer = speechsdk.SpeechRecognizer(speech_config=speech_config, audio_config=audio_config)

# Intentar reconocer audio del micro. Si no se detecta, usar archivo de prueba
try:
    print("Hable algo (o deje de hablar para finalizar)...")
    result = recognizer.recognize_once_async().get()
    
    if result.reason == speechsdk.ResultReason.RecognizedSpeech:
        print(f"Texto reconocido: {result.text}")
    elif result.reason == speechsdk.ResultReason.NoMatch:
        print("No se reconoció ningún discurso. Usando archivo de prueba...")
        # Cambiar a archivo de audio de prueba
        audio_config = speechsdk.audio.AudioConfig(filename="salida_hola.wav")
        recognizer = speechsdk.SpeechRecognizer(speech_config=speech_config, audio_config=audio_config)
        result = recognizer.recognize_once_async().get()
        if result.reason == speechsdk.ResultReason.RecognizedSpeech:
            print(f"Texto reconocido del archivo: {result.text}")
        else:
            print(f"Falló con archivo: {result.reason}")
    else:
        print(f"Reconocimiento falló: {result.reason}")
except Exception as e:
    print(f"Error: {e}")

 Probando SDK Speech (Speech-to-Text)...
Hable algo (o deje de hablar para finalizar)...
No se reconoció ningún discurso. Usando archivo de prueba...
Texto reconocido del archivo: Hola Estes una prevatete seconesedeca.


### 2.2 TTS API